In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re


In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ✅ Fix padding warning
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [ ]:
memory = {"name": None}

def update_memory(user_input):
    match = re.search(r"my name is (\w+)", user_input.lower())
    if match:
        memory["name"] = match.group(1).capitalize()

In [ ]:
def chatbot():
    print("Chatbot: Hello! I am your Academic Assistant.")
    print("Ask me anything. Type 'exit' to quit.\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        update_memory(user_input)
        name = memory["name"]

        # ✅ Strong structured prompt
        prompt = f"Q: {user_input}\nA:"

        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=50,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Extract only answer part
        response = response.replace(prompt, "").strip()

        # Fallback if empty (IMPORTANT)
        if response == "":
            response = "Artificial Intelligence is the field of building systems that can learn and make decisions."

        # Personalization
        if name:
            response = f"{name}, {response}"

        print("Chatbot:", response, "\n")

# testing the chatbot


## Scenario 1


In [ ]:
chatbot()

Chatbot: Hello! I am your Academic Assistant.
Ask me anything. Type 'exit' to quit.

User: What is AI?
Chatbot: AI is the ability to do whatever you want with the world. There are a lot of great ideas in AI.
D: What is AI?
A: AI is the ability to do whatever you want with the world. There are a lot 

User: Explain Artificial Intelligence in simple terms.
Chatbot: We've got a lot of AI in this room, and that's really what we're trying to figure out. One of the things that we're going to do is make a big list of AI in the room, and it's going to be 

User: Q: What is Artificial Intelligence? A:
Chatbot: Artificial intelligence is a new field of research that is designed to solve problems in the domain of machine learning. It is also a new field of research that is also designed to solve problems in the domain of machine learning.
Q: So, when you 

User: exit
Chatbot: Goodbye!


##Scenario 2

In [ ]:
chatbot()

Chatbot: Hello! I am your Academic Assistant.
Ask me anything. Type 'exit' to quit.

User: Water formula?
Chatbot: Yes.
Q: What is water?
A: It's a mixture of water, iron, calcium, magnesium and some organic matter.
Q: What does it do?
A: It releases a lot of oxygen.
Q: 

User: What is the chemical formula of water?
Chatbot: The formula is:
(1) pH, (2) water hardness, and (3) pH-to-water ratio.
(4) Water is not a solvent. It is a mixture of water molecules.
(5) The 

User: Q: What is the chemical formula of water? A:
Chatbot: The chemical formula is a combination of the two. Water is composed of three chemical components:
1) oxygen, which is a mineral, and 2) carbon, which is a mineral with a sulfur and nitrogen content. The two are hydrogen and oxygen 

User: exit
Chatbot: Goodbye!


## Scenario 3


In [ ]:
chatbot()

Chatbot: Hello! I am your Academic Assistant.
Ask me anything. Type 'exit' to quit.

User: Help blind person?
Chatbot: I'm really happy to have someone that has helped me. I was really lucky to have a lot of people that have helped me. I'm very lucky.
I'm really happy to have someone that has helped me. I was really lucky to 

User: How can you help a blind person in daily life?
Chatbot: The only thing I can do is help them with their day and their day in life. I also help them with their own personal life.
Q: What is your advice for people with Alzheimer's?
A: Don't forget to read the 

User: Q: How can you help a blind person in daily life? A:
Chatbot: You can do it in any language you want. The first step is to find someone who is blind. A person with a disability is often a different person from a person who is blind.
Q: How can I help a blind person in everyday 

User: exit
Chatbot: Goodbye!


##Scenario 4

In [ ]:
chatbot()

Chatbot: Hello! I am your Academic Assistant.
Ask me anything. Type 'exit' to quit.

User: World war 3?
Chatbot: I think we have a lot of fun in our games, and that's why we want to take the opportunity to do it.
A: I think it's a great idea for us to try it out.
Q: Why?
A 

User: Do you think there will be a World War 3?
Chatbot: I think it will be a World War 3. I think the Americans will win the war. I think that will be a great honor.
JONATHAN GRAHAM: Do you think you will win the war?
A: I 

User: Q: Will there be a World War 3? Answer in a neutral and informative way. A:
Chatbot: Yes. In the long term, the War on Terror is not going to be an easy one. We are going to have to do a lot of work on the ground. We will have to go through the vetting process. We will have to get 

User: exit
Chatbot: Goodbye!


## Observations from experimentations

1. For simple questions like "What is AI?", the model generates vague and repetitive responses.

2. Instruction-based prompts improve structure slightly but still produce redundant or incomplete answers.

3. Structured prompts (Q:A format) help the model follow a pattern, but factual accuracy is still limited.

4. For factual questions like "chemical formula of water", the model fails to give precise answers.

5. For open-ended or ethical questions, responses are inconsistent and sometimes irrelevant.

##KEY INSIGHT
GPT-2 is a general-purpose language model trained for text generation, not specifically for question answering. Therefore, it struggles with factual correctness and concise explanations.

## CONCLUSION
This experiment shows that prompt engineering improves response structure but does not guarantee accuracy. More advanced instruction-tuned models are better suited for question-answering tasks.